In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
# @title Install neccessary packages
!pip install -q transformers torch sentencepiece pillow fugashi unidic-lite

In [ ]:
# @title Load AI model
from transformers import ViTImageProcessor, AutoTokenizer, VisionEncoderDecoderModel
import torch
from PIL import Image
import cv2
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "kha-white/manga-ocr-base"

print(f"Đang tải model {model_name} lên {device}...")
try:
    processor = ViTImageProcessor.from_pretrained(model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = VisionEncoderDecoderModel.from_pretrained(model_name).to(device)
    print(f"Model loaded successfully!")
except Exception as e:
    print(f"Lỗi load model: {e}")

def get_text_with_ai(image_cv2):
    # Chuyển từ OpenCV sang PIL
    img_rgb = cv2.cvtColor(image_cv2, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(img_rgb)

    # Tiền xử lý ảnh
    pixel_values = processor(images=pil_img, return_tensors="pt").pixel_values.to(device)

    # Tạo văn bản
    output_ids = model.generate(pixel_values)
    text = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]
    return text.strip()

In [ ]:
# @title Recognize title and change file name
import os
import re
import cv2
import io
import torch
import numpy as np
from PIL import Image

# --- 1. CONFIGURATION ---
YEAR = "2026" # @param {type:"string"}
DRIVE_URL = "https://drive.google.com/drive/folders/1RcVHtflAWN1fTUiGngy9mNVrol7t9-Q2" # @param {type:"string"}

def clean_ai_text(text):
    text = re.sub(r'\s+', '', text)
    prefixes_to_remove = ["・", "・・", "| ", "# ", "さて", "そして", "でも", "それでも", "この", "そう", "そうです", "そうですね", "また"]
    for pref in prefixes_to_remove:
        if text.startswith(pref):
            text = text[len(pref):]
    return text

def enhance_image_v2(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
    gray = clahe.apply(gray)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thresh = cv2.medianBlur(thresh, 3)
    return thresh

# --- 2. MAIN PROCESSING LOGIC ---
def process_shared_folder(url):
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    service = build('drive', 'v3')

    match = re.search(r'folders/([a-zA-Z0-9_-]+)', url)
    if not match: return

    folder_id = match.group(1)
    query = f"'{folder_id}' in parents and (mimeType = 'image/jpeg' or mimeType = 'image/png') and trashed = false"
    results = service.files().list(q=query, fields="files(id, name)", orderBy="name").execute()
    items = results.get('files', [])

    if not items: return

    print(f"Đang xử lý {len(items)} ảnh...")
    index_count = 0

    for item in items:
        file_id, old_name = item['id'], item['name']
        try:
            from googleapiclient.http import MediaIoBaseDownload
            request = service.files().get_media(fileId=file_id)
            fh = io.BytesIO()
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done: _, done = downloader.next_chunk()

            nparr = np.frombuffer(fh.getvalue(), np.uint8)
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            h, w = img.shape[:2]

            new_name = None

            # --- BƯỚC 1: KIỂM TRA TRANG BÌA (表紙) ---
            raw_cover = get_text_with_ai(img[int(h*0.02):int(h*0.15), int(w*0.0):int(w*0.75)])
            title_cover = re.sub(r'\s+', '', raw_cover)
            if "職場の教養" in title_cover:
                new_name = "00_表紙.jpg"

            # --- BƯỚC 2: KIỂM TRA MỤC LỤC ---
            if not new_name:
                raw_index = get_text_with_ai(img[int(h*0.02):int(h*0.13), int(w*0.5):int(w*0.99)])
                title_index = re.sub(r'\s+', '', raw_index)
                # print(f"Title {title_index}")

                if "職場の教養" in title_index or title_index == "そういうことで,":
                    index_count += 1
                    new_name = f"01_目次_{index_count}.jpg"

            # --- BƯỚC 3: KIỂM TRA TRANG HÀNG NGÀY ---
            if not new_name:
                date_crop = enhance_image_v2(img[int(h*0.03):int(h*0.12), int(w*0.08):int(w*0.30)])
                date_raw = get_text_with_ai(cv2.cvtColor(date_crop, cv2.COLOR_GRAY2BGR))
                nums = re.findall(r'\d+', date_raw)

                if len(nums) >= 2:
                    m_str, d_str = nums[-2].zfill(2), nums[-1].zfill(2)
                    title_daily = clean_ai_text(get_text_with_ai(img[int(h*0.03):int(h*0.13), int(w*0.33):int(w*0.95)]))
                    safe_title = re.sub(r'[^぀-ゟ゠-ヿ一-鿿]', '', title_daily)
                    new_name = f"{YEAR}{m_str}{d_str}.{safe_title if safe_title else 'Daily'}.jpg"

            if new_name:
                service.files().update(fileId=file_id, body={'name': new_name}).execute()
                print(f"Processed: {old_name} -> {new_name}")
            else:
                print(f"Could not determine name for: {old_name}")
        except Exception as e:
            print(f"Lỗi file {old_name}: {e}")

process_shared_folder(DRIVE_URL)

In [ ]:
if False:
    # @title Debug tool: Draw frame on daily's date and title
    import cv2
    import os
    import re
    from google.colab.patches import cv2_imshow
    import numpy as np

    def test_recognition_v2(image_path):
        if not os.path.exists(image_path):
            print(f"File not found: {image_path}")
            return

        img = cv2.imread(image_path)
        h, w = img.shape[:2]

        # Tỉ lệ vùng Ngày tháng
        dx1, dx2 = int(w * 0.08), int(w * 0.30)
        dy1, dy2 = int(h * 0.03), int(h * 0.12)

        # Tỉ lệ vùng Tiéu đề (35% để an toàn cho chữ cái đầu)
        tx1, tx2 = int(w * 0.30), int(w * 0.95)
        ty1, ty2 = int(h * 0.03), int(h * 0.13)

        date_crop = img[dy1:dy2, dx1:dx2]
        title_crop = img[ty1:ty2, tx1:tx2]

        # --- 1. HIỂN THỊ KHUNG QUÉT TRÊN ẢNH GỐC ---
        display_img = img.copy()
        cv2.rectangle(display_img, (dx1, dy1), (dx2, dy2), (0, 255, 0), 10) # Xanh lá = Ngày
        cv2.rectangle(display_img, (tx1, ty1), (tx2, ty2), (0, 0, 255), 10) # Đỏ = Tiêu đề
        print("--- 1. VÙNG QUÉT TỔNG THỂ (Xanh=Ngày, Đỏ=Tiêu đề) ---")
        cv2_imshow(cv2.resize(display_img, (int(w*0.3), int(h*0.3))))

        print("\n--- 2. VÙNG QUÉT NGÀY (Trắng đen) ---")
        gray = cv2.cvtColor(date_crop, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=4.0, tileGridSize=(8,8))
        gray = clahe.apply(gray)
        _, date_bin = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        date_bin = cv2.medianBlur(date_bin, 3)
        cv2_imshow(date_bin)

        print("\n--- 3. VÙNG TIÊU ĐỀ (Ảnh màu gửi cho AI) ---")
        cv2_imshow(title_crop)

        # Chạy thử AI
        date_bin_bgr = cv2.cvtColor(date_bin, cv2.COLOR_GRAY2BGR)
        date_raw = get_text_with_ai(date_bin_bgr)
        nums = re.findall(r'\d+', date_raw)

        title_raw = get_text_with_ai(title_crop)
        title_cleaned = clean_ai_text(title_raw)

        print(f"\n[AI Raw Date]: {date_raw} -> Số tìm thấy: {nums}")
        print(f"[AI Raw Title]: {title_raw}")
        print(f"[Title sau khi dọn dẹp]: {title_cleaned}")

    # Chạy test vối file tạm thồi có trong content
    import glob
    files = glob.glob('/content/16041236.JPG')
    if files:
        test_recognition_v2(files[0])
    else:
        print("Không tìm thấy file ảnh nào trong /content/ để test.")

In [ ]:
if False:
    # @title Debug tool: Draw frame on the picture
    import cv2
    import os
    from google.colab.patches import cv2_imshow

    def draw_consistent_debug(image_path, x1_pct, x2_pct, y1_pct, y2_pct):
        if not os.path.exists(image_path):
            print(f"File not found: {image_path}")
            return

        img = cv2.imread(image_path)
        h, w = img.shape[:2]

        # Tính toán pixel dựa trên % (Giống logic Cell 4)
        x1, x2 = int(w * x1_pct), int(w * x2_pct)
        y1, y2 = int(h * y1_pct), int(h * y2_pct)

        display_img = img.copy()

        # Vẽ khung debug (Màu đỏ)
        cv2.rectangle(display_img, (x1, y1), (x2, y2), (0, 0, 255), 10)

        print(f"Kích thước ảnh: {w}x{h}")
        print(f"Vùng quét đang thử nghiệm:")
        print(f"X: {x1_pct*100}% -> {x2_pct*100}% (Pixel: {x1} -> {x2})")
        print(f"Y: {y1_pct*100}% -> {y2_pct*100}% (Pixel: {y1} -> {y2})")

        # Hiển thị kết quả
        cv2_imshow(cv2.resize(display_img, (int(w*0.3), int(h*0.3))))

        print("\n--- VÙNG CẮT THỰC TẾ ---")
        crop = img[y1:y2, x1:x2]
        cv2_imshow(crop)

    # --- CẤU HÌNH (Sử dụng hệ số thập phân giống Cell 4: 0.33 = 33%) ---
    TEST_IMAGE = "/content/16041203.JPG"

    # Ví dụ: Vùng tiêu đề của 表紙, chứa chữ 職場の教養
    DX1, DX2 = 0.0, 0.75
    DY1, DY2 = 0.02, 0.15

    # Ví dụ: Vùng tiêu đề của 目次1, chứa chữ 職場の教養
    DX1, DX2 = 0.5, 0.99
    DY1, DY2 = 0.02, 0.13

    draw_consistent_debug(TEST_IMAGE, DX1, DX2, DY1, DY2)